# ST2 — Geopolitical Risk: Data Ingestion & Cleaning
**Team 7 Lambda | Phase 1 | Feb 27 – Mar 2**

**Goal:** Pull and clean four time series:
1. GPR Index (Iacoviello) — monthly geopolitical risk
2. NY Fed GSCPI — global supply chain pressure
3. FRED macro series — CPI, yield curve, oil price, industrial production
4. Sector ETFs — XLK, XLE, XLB, SOXX via yfinance

**Key constraint:** All joins use `pd.merge_asof` backward direction to prevent look-ahead bias.

**Output:** `data/processed/ST2/st2_gpr.parquet`, `data/processed/ST2/st2_gscpi.parquet`, `data/processed/ST2/st2_macro.parquet`, `data/processed/ST2/st2_etfs.parquet`, `data/processed/ST2/st2_master.parquet`


In [1]:
import sys
sys.path.insert(0, '../src')

import importlib.util
import pandas as pd
import numpy as np
import requests
from pathlib import Path

from config import (FRED_API_KEY, SECTOR_ETFS, ST2_PROC, ST2_RAW,
                    START_YEAR, END_YEAR, GPR_SMOOTH_WINDOW)
from st2_pipeline import save_st2_master
from utils import save_parquet, set_style, log, KEY_EVENTS

set_style()
log.info('Setup complete.')


15:50:24 [INFO] Setup complete.


## 2A — GPR Index (Iacoviello)

**Source:** https://www.matteoiacoviello.com/gpr.htm  
**Download:** Click 'Data (Excel file)' → save as `data/raw/gpr/gpr_data.xlsx`  
Contains: `gpr` (composite), `gprt` (threats), `gpra` (acts), monthly from 1985, plus 18 country indices.

In [2]:
# ── GPR Index ──────────────────────────────────────────────────────────────

GPR_PATH = next(
    (p for p in [ST2_RAW / 'data_gpr_export.xlsx', ST2_RAW / 'data_gpr_export.xls'] if p.exists()),
    ST2_RAW / 'data_gpr_export.xlsx'
)

def load_gpr(path: Path) -> pd.DataFrame:
    """
    Load Iacoviello GPR Excel file.
    Returns monthly DataFrame with columns: date, gpr, gprt, gpra,
    gpr_smooth (3-month rolling mean), plus country-specific GPR columns.
    """
    if not path.exists():
        log.warning(f'GPR file not found at {path}. Download from matteoiacoviello.com')
        return pd.DataFrame()

    suffix = path.suffix.lower()
    if suffix == '.xlsx':
        df = pd.read_excel(path, sheet_name=0)
    elif suffix == '.xls':
        if importlib.util.find_spec('xlrd') is None:
            log.error(
                'GPR source is .xls but xlrd is not installed in this environment. '
                'Open the file in Excel/Numbers and resave it as data_gpr_export.xlsx in data/raw/ST2/.'
            )
            return pd.DataFrame()
        df = pd.read_excel(path, sheet_name=0, engine='xlrd')
    else:
        log.error(f'Unsupported GPR file format: {path.name}')
        return pd.DataFrame()

    # GPR Excel: first sheet has monthly data, columns include date + indices
    log.info(f'GPR raw columns: {list(df.columns[:10])}...')
    
    # Normalize column names
    df.columns = df.columns.str.strip().str.lower()
    
    # Find date column (usually 'month' or 'date' or first column)
    date_col = next((c for c in df.columns if c in ['month', 'date', 'year_month']), df.columns[0])
    df = df.rename(columns={date_col: 'date'})
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date']).sort_values('date')
    
    # Ensure core columns exist (case-insensitive)
    col_map = {c: c.lower() for c in df.columns}
    df = df.rename(columns=col_map)
    
    # Apply 3-month rolling mean to smooth noise (no look-ahead: rolling uses past values only)
    for col in ['gpr', 'gprt', 'gpra']:
        if col in df.columns:
            df[f'{col}_smooth'] = df[col].rolling(GPR_SMOOTH_WINDOW, min_periods=1).mean()
    
    # Forward-fill any gaps (holiday delays) — max 2 consecutive
    gpr_cols = [c for c in df.columns if c.startswith('gpr')]
    df[gpr_cols] = df[gpr_cols].fillna(method='ffill', limit=2)
    
    # Filter to analysis window
    df = df[(df['date'].dt.year >= START_YEAR) & (df['date'].dt.year <= END_YEAR)]
    
    log.info(f'GPR loaded: {df.shape} | {df["date"].min()} → {df["date"].max()}')
    return df.reset_index(drop=True)


gpr_df = load_gpr(GPR_PATH)

if not gpr_df.empty:
    display(gpr_df[['date', 'gpr', 'gprt', 'gpra', 'gpr_smooth']].head(10))

15:50:24 [INFO] GPR raw columns: ['month', 'GPR', 'GPRT', 'GPRA', 'GPRH', 'GPRHT', 'GPRHA', 'SHARE_GPR', 'N10', 'SHARE_GPRH']...


/var/folders/pj/ybxvss_x0sbdswwqtvfjh5s80000gn/T/ipykernel_81138/528992568.py:56: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[gpr_cols] = df[gpr_cols].fillna(method='ffill', limit=2)
15:50:24 [INFO] GPR loaded: (156, 118) | 2010-01-01 00:00:00 → 2022-12-01 00:00:00


,date,gpr,gprt,gpra,gpr_smooth
0,2010-01-01,91.581024,84.972992,100.410942,88.876567
1,2010-02-01,80.725357,78.846275,80.711739,88.417765
2,2010-03-01,74.116943,73.496910,70.039345,82.141108
3,2010-04-01,88.761581,92.056915,81.268402,81.201294
4,2010-05-01,88.958710,87.494110,85.857391,83.945745
5,2010-06-01,96.453697,110.691719,70.988625,91.391329
6,2010-07-01,79.381378,75.185379,84.048233,88.264595
7,2010-08-01,80.993866,71.951530,85.982574,85.609647
8,2010-09-01,71.168549,68.947540,69.293953,77.181264
9,2010-10-01,65.991936,62.096180,76.168915,72.718117


## 2B — NY Fed Global Supply Chain Pressure Index (GSCPI)

**Source:** https://www.newyorkfed.org/research/policy/gscpi  
**Download:** Click 'Download data' (Excel) → save as `data/raw/nyfed/gscpi.xlsx`  
Monthly from January 1997. Positive values = above-average supply chain pressure.

In [3]:
# ── GSCPI ──────────────────────────────────────────────────────────────────

GSCPI_PATH = next(
    (
        p
        for p in [
            ST2_RAW / 'gscpi.csv',
            ST2_RAW / 'gscpi.xlsx',
            ST2_RAW / 'gscpi_data.xlsx',
            ST2_RAW / 'gscpi_data.xls',
            Path.home() / 'Downloads' / 'gscpi_data.xlsx',
            Path.home() / 'Downloads' / 'gscpi_data.xls',
        ]
        if p.exists()
    ),
    ST2_RAW / 'gscpi.csv'
)

def load_gscpi(path: Path) -> pd.DataFrame:
    """
    Load NY Fed GSCPI source.
    Returns: date (monthly) | gscpi (standardized index)
    Positive values = above-average supply chain pressure.
    """
    if not path.exists():
        log.warning(f'GSCPI not found at {path}. Download from NY Fed.')
        return pd.DataFrame()

    if path.suffix.lower() == '.csv':
        df = pd.read_csv(path)
    else:
        df = None
        # Try common sheet names
        for sheet in ['GSCPI Monthly Data', 'GSCPI', 'Sheet1', 0]:
            try:
                candidate = pd.read_excel(path, sheet_name=sheet, header=0)
                if not candidate.empty:
                    df = candidate
                    break
            except Exception:
                continue

        if df is None:
            log.error(f'Could not read any GSCPI sheet from {path.name}')
            return pd.DataFrame()

    df.columns = df.columns.str.strip().str.lower()

    # Date column
    date_col = next((c for c in df.columns if 'date' in c or 'month' in c or 'period' in c), df.columns[0])
    df = df.rename(columns={date_col: 'date'})
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

    # GSCPI column
    gscpi_col = next((c for c in df.columns if 'gscpi' in c or 'index' in c or 'pressure' in c), None)
    if gscpi_col is None:
        if len(df.columns) < 2:
            log.error(f'No GSCPI value column found in {path.name}')
            return pd.DataFrame()
        gscpi_col = df.columns[1]

    df = df.rename(columns={gscpi_col: 'gscpi'})
    df['gscpi'] = pd.to_numeric(df['gscpi'], errors='coerce')

    df = df.dropna(subset=['date', 'gscpi']).sort_values('date')
    df = df[(df['date'].dt.year >= START_YEAR) & (df['date'].dt.year <= END_YEAR)]

    log.info(f'GSCPI loaded: {df.shape} | {df["date"].min()} → {df["date"].max()}')
    return df[['date', 'gscpi']].reset_index(drop=True)


gscpi_df = load_gscpi(GSCPI_PATH)

if not gscpi_df.empty:
    display(gscpi_df.head())



15:50:24 [INFO] GSCPI loaded: (156, 4) | 2010-01-31 00:00:00 → 2022-12-31 00:00:00


,date,gscpi
0,2010-01-31,-0.319626
1,2010-02-28,-0.151996
2,2010-03-31,0.367305
3,2010-04-30,0.298844
4,2010-05-31,0.438048


## 2C — FRED Macro Series

**Source:** FRED API — https://fred.stlouisfed.org/docs/api/fred/  
**Setup:** Get a free API key at https://fred.stlouisfed.org/docs/api/api_key.html  
Add to `.env` as `FRED_API_KEY=your_key_here`

In [4]:
# ── FRED API ───────────────────────────────────────────────────────────────

from config import FRED_SERIES

def fetch_fred_series(series_id: str, api_key: str) -> pd.DataFrame:
    """
    Fetch a single FRED series via their public API.
    Returns: date | value
    No external library needed — pure requests.
    """
    url = 'https://api.stlouisfed.org/fred/series/observations'
    params = {
        'series_id':         series_id,
        'api_key':           api_key,
        'file_type':         'json',
        'observation_start': f'{START_YEAR}-01-01',
        'observation_end':   f'{END_YEAR}-12-31',
    }
    r = requests.get(url, params=params, timeout=15)
    r.raise_for_status()
    
    obs = r.json().get('observations', [])
    df = pd.DataFrame(obs)
    df['date']  = pd.to_datetime(df['date'])
    df['value'] = pd.to_numeric(df['value'], errors='coerce')  # '.' = missing in FRED
    df = df[['date', 'value']].dropna().sort_values('date')
    log.info(f'FRED {series_id}: {len(df)} observations')
    return df


def fetch_all_fred(api_key: str) -> pd.DataFrame:
    """
    Fetch all series from FRED_SERIES config dict.
    Resamples daily series to monthly (end-of-month, last value).
    Returns single wide DataFrame indexed by date (monthly).
    """
    frames = {}
    for name, sid in FRED_SERIES.items():
        try:
            s = fetch_fred_series(sid, api_key).set_index('date')['value']
            # Resample to month-end for alignment
            frames[name] = s.resample('ME').last()
        except Exception as e:
            log.warning(f'Failed to fetch {sid}: {e}')
    
    if not frames:
        return pd.DataFrame()
    
    macro_df = pd.DataFrame(frames).sort_index()
    # Forward-fill up to 2 months (handles quarterly GDP series gaps)
    macro_df = macro_df.fillna(method='ffill', limit=2)
    log.info(f'FRED combined: {macro_df.shape}')
    return macro_df.reset_index()


if FRED_API_KEY:
    macro_df = fetch_all_fred(FRED_API_KEY)
    display(macro_df.head())
else:
    log.warning('No FRED_API_KEY found in .env — skipping FRED fetch.')
    log.warning('Get a free key at https://fred.stlouisfed.org/docs/api/api_key.html')
    macro_df = pd.DataFrame()

15:50:25 [INFO] FRED CPIAUCSL: 156 observations


15:50:25 [INFO] FRED T10Y2Y: 3252 observations


15:50:25 [INFO] FRED DCOILWTICO: 3267 observations


15:50:25 [WARNING] Failed to fetch GOLDPMGBD228NLBM: 400 Client Error: Bad Request for url: https://api.stlouisfed.org/fred/series/observations?series_id=GOLDPMGBD228NLBM&api_key=7d5602281cab3ede91435cc93577edba&file_type=json&observation_start=2010-01-01&observation_end=2022-12-31


15:50:25 [INFO] FRED A191RL1Q225SBEA: 52 observations


15:50:25 [INFO] FRED INDPRO: 156 observations


/var/folders/pj/ybxvss_x0sbdswwqtvfjh5s80000gn/T/ipykernel_81138/3757171935.py:51: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  macro_df = macro_df.fillna(method='ffill', limit=2)
15:50:25 [INFO] FRED combined: (156, 5)


,date,CPI,YIELD_CURVE,OIL_PRICE,GDP_GROWTH,INDUSTRIAL_PROD
0,2010-01-31,217.488,2.81,72.85,1.9,89.3426
1,2010-02-28,217.281,2.80,79.72,1.9,89.6779
2,2010-03-31,217.353,2.82,83.45,1.9,90.2928
3,2010-04-30,217.403,2.72,86.07,3.9,90.5991
4,2010-05-31,217.290,2.55,74.00,3.9,91.8230


## 2D — Sector ETF Prices (yfinance)

**Source:** Yahoo Finance via yfinance library  
**No API key needed.** Pulls daily OHLCV for XLK, XLE, XLB, SOXX.

In [5]:
# ── ETF Prices via yfinance ────────────────────────────────────────────────

import yfinance as yf

def fetch_etfs(tickers: dict[str, str], start: str, end: str) -> pd.DataFrame:
    """
    Download adjusted close prices for multiple ETFs.
    Resamples daily → monthly returns (last trading day of month).
    Returns wide DataFrame: date | XLK | XLE | XLB | SOXX
    """
    symbols = list(tickers.values())
    raw = yf.download(symbols, start=start, end=end, auto_adjust=True, progress=False)

    if raw is None or raw.empty:
        raise RuntimeError('yfinance returned no ETF data')

    # Extract Close prices
    prices = raw['Close'] if isinstance(raw.columns, pd.MultiIndex) else raw

    # Resample to monthly (last trading day)
    monthly = prices.resample('ME').last()

    # Compute monthly returns
    returns = monthly.pct_change().add_suffix('_return')

    result = pd.concat([monthly, returns], axis=1).reset_index()
    result.columns.name = None
    log.info(f'ETF data: {result.shape} | {result["Date"].min()} → {result["Date"].max()}')
    return result.rename(columns={'Date': 'date'})


etf_df = fetch_etfs(
    SECTOR_ETFS,
    start=f'{START_YEAR}-01-01',
    end=f'{END_YEAR}-12-31'
)
display(etf_df.head())



15:50:26 [INFO] Failed to create TzCache, reason: Error creating TzCache folder: '/Users/chaitanya/Library/Caches/py-yfinance' reason: [Errno 17] File exists: '/Users/chaitanya/Library/Caches/py-yfinance'. TzCache will not be used. Tip: You can direct cache to use a different location with 'set_tz_cache_location(mylocation)'


15:50:26 [INFO] Failed to create TzCache, reason: Error creating TzCache folder: '/Users/chaitanya/Library/Caches/py-yfinance' reason: [Errno 17] File exists: '/Users/chaitanya/Library/Caches/py-yfinance'. TzCache will not be used. Tip: You can direct cache to use a different location with 'set_tz_cache_location(mylocation)'


15:50:26 [INFO] Failed to create TzCache, reason: Error creating TzCache folder: '/Users/chaitanya/Library/Caches/py-yfinance' reason: [Errno 17] File exists: '/Users/chaitanya/Library/Caches/py-yfinance'. TzCache will not be used. Tip: You can direct cache to use a different location with 'set_tz_cache_location(mylocation)'


15:50:27 [INFO] ETF data: (156, 13) | 2010-01-31 00:00:00 → 2022-12-31 00:00:00


,date,SOXX,XLB,XLE,XLI,XLK,XLU,SOXX_return,XLB_return,XLE_return,XLI_return,XLK_return,XLU_return
0,2010-01-31,12.091352,10.693369,16.107283,20.285154,8.458262,8.501966,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-02-28,13.056297,11.175879,16.591980,21.366837,8.752845,8.389676,0.079805,0.045122,0.030092,0.053324,0.034828,-0.013207
2,2010-03-31,13.903087,12.060581,17.059359,23.224243,9.351522,8.615406,0.064857,0.079162,0.028169,0.086929,0.068398,0.026906
3,2010-04-30,14.184410,12.085474,17.768192,24.212982,9.468920,8.841976,0.020235,0.002064,0.041551,0.042574,0.012554,0.026298
4,2010-05-31,13.256038,10.933460,15.733641,22.005041,8.760470,8.353984,-0.065450,-0.095322,-0.114505,-0.091188,-0.074818,-0.055190


In [6]:
# ── Merge all ST2 series into one master monthly DataFrame ─────────────────
master = save_st2_master(
    ST2_PROC / 'st2_master.parquet',
    gpr_df=gpr_df,
    gscpi_df=gscpi_df,
    macro_df=macro_df,
    etf_df=etf_df,
)

if not master.empty:
    display(master.describe())
else:
    log.warning('No ST2 data loaded. Check that files exist in data/raw/')

# ── Save outputs ───────────────────────────────────────────────────────────
if not gpr_df.empty:   save_parquet(gpr_df,    ST2_PROC / 'st2_gpr.parquet',    'GPR')
if not gscpi_df.empty: save_parquet(gscpi_df,  ST2_PROC / 'st2_gscpi.parquet',  'GSCPI')
if not macro_df.empty: save_parquet(macro_df,  ST2_PROC / 'st2_macro.parquet',  'FRED Macro')
if not etf_df.empty:   save_parquet(etf_df,    ST2_PROC / 'st2_etfs.parquet',   'ETFs')

log.info('Phase 1 ST2 complete.')


/Users/chaitanya/Desktop/ADV_Project/vaartha/notebooks/../src/st2_pipeline.py:69: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  master[value_cols] = master[value_cols].ffill(limit=2)
15:50:27 [INFO] Built ST2 master table: 156 rows x 136 cols


15:50:27 [INFO] Saved ST2 Master: 156 rows x 136 cols -> /Users/chaitanya/Desktop/ADV_Project/vaartha/data/processed/st2_master.parquet


,date,gpr,gprt,gpra,gprh,gprht,gprha,share_gpr,n10,share_gprh,...,XLE,XLI,XLK,XLU,SOXX_return,XLB_return,XLE_return,XLI_return,XLK_return,XLU_return
count,156,156.000000,156.000000,156.000000,156.000000,156.000000,156.000000,156.000000,156.000000,156.000000,...,156.000000,156.000000,156.000000,156.000000,155.000000,155.000000,155.000000,155.000000,155.000000,155.000000
mean,2016-06-16 01:50:46.153846272,96.732694,106.409479,85.263827,77.737161,98.940803,64.592113,2.901385,23567.019231,2.804310,...,23.415206,53.512321,29.757266,18.680853,0.016827,0.009523,0.009118,0.011310,0.014069,0.009418
min,2010-01-01 00:00:00,58.420769,59.019047,28.454628,46.875725,55.361584,21.128803,1.752263,16131.000000,1.691007,...,11.529376,20.285154,8.287704,8.295333,-0.178349,-0.164795,-0.343709,-0.186259,-0.119671,-0.112810
25%,2013-03-24 06:00:00,79.411556,81.005175,64.251028,66.039206,78.409691,47.972184,2.381858,21346.250000,2.382315,...,20.865838,33.039972,13.025973,12.374928,-0.023805,-0.020434,-0.033398,-0.017741,-0.015173,-0.012131
50%,2016-06-16 00:00:00,90.091309,95.498116,81.459671,73.301750,91.384418,62.842768,2.702184,23327.000000,2.644306,...,23.186625,48.045721,20.437757,18.176648,0.025428,0.011207,0.015539,0.011335,0.018553,0.012295
75%,2019-09-08 12:00:00,105.345043,119.549604,100.199532,86.794209,111.978848,78.630545,3.159702,26315.250000,3.131036,...,25.560452,68.387920,38.179835,24.311754,0.062243,0.040588,0.047015,0.041753,0.045235,0.036128
max,2022-12-01 00:00:00,318.954926,403.713623,250.955856,167.344406,264.452179,167.498856,9.566683,30414.000000,6.036825,...,40.754524,99.322769,84.243073,33.393940,0.188526,0.173365,0.307639,0.160274,0.137364,0.105520
std,NaN,29.552307,42.367842,32.632826,18.429126,29.793113,23.577571,0.886387,3265.752329,0.664817,...,5.162075,22.543543,20.873969,7.080196,0.067723,0.055820,0.081519,0.052668,0.050326,0.040063


15:50:27 [INFO] Saved GPR: 156 rows × 118 cols → /Users/chaitanya/Desktop/ADV_Project/vaartha/data/processed/st2_gpr.parquet


15:50:27 [INFO] Saved GSCPI: 156 rows × 2 cols → /Users/chaitanya/Desktop/ADV_Project/vaartha/data/processed/st2_gscpi.parquet


15:50:27 [INFO] Saved FRED Macro: 156 rows × 6 cols → /Users/chaitanya/Desktop/ADV_Project/vaartha/data/processed/st2_macro.parquet


15:50:27 [INFO] Saved ETFs: 156 rows × 13 cols → /Users/chaitanya/Desktop/ADV_Project/vaartha/data/processed/st2_etfs.parquet


15:50:27 [INFO] Phase 1 ST2 complete.
